# KURE 번아웃 스타일 트랜스퍼 v1
**목적**: 대화체 학습 데이터 → 일기체 변환으로 도메인 미스매치 해결

## 배경
- 현재 모델은 **대화체** 말뭉치로 학습됨
- 실제 사용자는 **일기체**로 작성
- → 도메인 미스매치 → Stage 2 F1 47.54% 한계

## 파이프라인
```
stage2_train_v3.csv (대화체, 라벨 있음)
        ↓  EEVE-Korean-Instruct (Apache 2.0)
stage2_train_diary.csv (일기체, 라벨 그대로)
        ↓
원본 + 일기체 혼합 → Stage 2 재학습
        ↓
F1 47.54% 대비 성능 비교
```

## 단계별 구성
| 단계 | 내용 |
|------|------|
| 1~3 | 환경 설정, 모델 로드 |
| 4 | 프롬프트 테스트 (소량) → 사람이 품질 확인 |
| 5~6 | 전체 변환 + 체크포인트 저장 |
| 7 | 품질 필터링 |
| 8~10 | Stage 2 재학습 + 성능 비교 |

## 1. 환경 설정

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate bitsandbytes sentence-transformers scikit-learn

In [ ]:
import os, json, time, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Burnout'

STAGE2_CATEGORIES = {0: '정서적_고갈', 1: '좌절_압박', 2: '부정적_대인관계', 3: '자기비하'}

# 입출력 경로
S2_TRAIN_PATH    = f'{DATA_PATH}/stage2_train_v3.csv'
S2_VAL_PATH      = f'{DATA_PATH}/stage2_val_v3.csv'
DIARY_TRAIN_PATH = f'{DATA_PATH}/stage2_train_diary.csv'   # 변환 결과 저장
CHECKPOINT_PATH  = f'{DATA_PATH}/st_checkpoint.json'       # 변환 진행 상황 저장
S2_MODEL_PATH    = f'{DATA_PATH}/stage2_model_v3.pt'       # warm-start용 기존 모델
SAVE_MODEL_PATH  = f'{DATA_PATH}/stage2_model_st.pt'       # 스타일 트랜스퍼 재학습 결과

print('경로 확인:')
for name, path in [('S2 Train', S2_TRAIN_PATH), ('S2 Val', S2_VAL_PATH),
                   ('S2 Model (warm-start)', S2_MODEL_PATH)]:
    print(f'  {"✅" if os.path.exists(path) else "❌"} {name}: {path}')

## 3. EEVE-Korean-Instruct 로드 (4-bit 양자화)

- 모델: `yanolja/EEVE-Korean-Instruct-10.8B-v1.0`
- 라이선스: **Apache 2.0**
- 4-bit 양자화 → Colab T4 (16GB)에서 약 6~7GB VRAM 사용

In [ ]:
MODEL_ID = 'yanolja/EEVE-Korean-Instruct-10.8B-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'모델 로딩 중: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
eeve = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
eeve.eval()
print('✅ EEVE 로드 완료')
print(f'   VRAM 사용: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 4. 프롬프트 설계 + 소량 테스트

**전체 변환 전에 반드시 이 셀을 실행해서 품질을 직접 확인하세요.**

In [ ]:
SYSTEM_PROMPT = """당신은 한국어 텍스트의 문체를 변환하는 전문가입니다.
대화체 문장을 일기체로 자연스럽게 바꿔주세요.

규칙:
1. 감정과 의미는 반드시 유지할 것
2. 문체만 일기 쓰듯 바꿀 것 (종결어미: ~했다, ~이었다, ~였다 등)
3. 1인칭 시점 유지
4. 원문보다 길어져도 되지만 너무 길지 않게 (1~3문장)
5. 변환 결과만 출력할 것 (설명 없이)"""


def convert_to_diary(text: str, max_new_tokens: int = 150) -> str:
    """대화체 → 일기체 변환"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"원문: {text}\n일기체:"}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt'
    ).to(eeve.device)

    with torch.no_grad():
        output_ids = eeve.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy (재현성)
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return result


# ── 소량 테스트 ──────────────────────────────────────
test_samples = [
    ('정서적_고갈', '오늘도 야근이었어. 집에 오니 아무것도 하기 싫고 그냥 쓰러지고 싶었어.'),
    ('좌절_압박',   '팀장이 또 내 앞에서 나를 무시했어. 너무 억울하고 화가 났어.'),
    ('부정적_대인관계', '요즘 들어 출근이 너무 싫어. 사람들 얼굴 보기도 싫고 그냥 다 피하고 싶어.'),
    ('자기비하',    '나는 왜 이것밖에 못 할까. 이러니 아무도 날 인정 안 하지.'),
]

print('=' * 60)
print('📝 소량 변환 테스트')
print('=' * 60)
print('⚠️  아래 결과를 직접 확인하고 품질이 괜찮으면 전체 변환을 진행하세요.\n')

for category, original in test_samples:
    converted = convert_to_diary(original)
    print(f'[{category}]')
    print(f'  원문 : {original}')
    print(f'  변환 : {converted}')
    print()

## 5. 전체 데이터 로드

In [ ]:
s2_train = pd.read_csv(S2_TRAIN_PATH)
s2_val   = pd.read_csv(S2_VAL_PATH)

print(f'Stage 2 Train: {len(s2_train):,}건')
print(f'Stage 2 Val  : {len(s2_val):,}건')
print()
print('Train 클래스 분포:')
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train['label'] == label).sum()
    pct = cnt / len(s2_train) * 100
    print(f'  {cat}: {cnt:,}건 ({pct:.1f}%)')

## 6. 전체 변환 실행 (체크포인트 저장)

> ⚠️ 시간이 많이 걸립니다 (건당 약 2~5초 × 전체 건수).
> 중간에 Colab이 끊겨도 `st_checkpoint.json`이 저장되어 이어서 실행 가능합니다.

In [ ]:
# ── 체크포인트 로드 (이전 진행분 이어받기) ──────────────
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    converted_rows = checkpoint['rows']
    start_idx = len(converted_rows)
    print(f'✅ 체크포인트 발견: {start_idx}건 이미 변환됨, 이어서 진행')
else:
    converted_rows = []
    start_idx = 0
    print('새로 시작')

SAVE_EVERY = 50   # 50건마다 체크포인트 저장
TOTAL = len(s2_train)

print(f'변환 대상: {TOTAL - start_idx:,}건 남음 (전체 {TOTAL:,}건)')
print()

for i in tqdm(range(start_idx, TOTAL), desc='스타일 변환'):
    row = s2_train.iloc[i]
    original_text = str(row['text'])
    label = int(row['label'])

    try:
        diary_text = convert_to_diary(original_text)
        # 변환 결과가 너무 짧거나 비어있으면 원문 사용
        if len(diary_text.strip()) < 5:
            diary_text = original_text
            status = 'fallback'
        else:
            status = 'ok'
    except Exception as e:
        diary_text = original_text
        status = f'error: {e}'

    converted_rows.append({
        'original': original_text,
        'text': diary_text,
        'label': label,
        'status': status,
    })

    # 체크포인트 저장
    if (i + 1) % SAVE_EVERY == 0 or i == TOTAL - 1:
        with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
            json.dump({'rows': converted_rows}, f, ensure_ascii=False)

print(f'\n✅ 변환 완료: {len(converted_rows):,}건')

# 상태 요약
status_counts = {}
for r in converted_rows:
    s = r['status'] if r['status'] in ('ok', 'fallback') else 'error'
    status_counts[s] = status_counts.get(s, 0) + 1
print(f'  ok      : {status_counts.get("ok", 0):,}건')
print(f'  fallback: {status_counts.get("fallback", 0):,}건 (원문 사용)')
print(f'  error   : {status_counts.get("error", 0):,}건')

## 7. 품질 확인 + 필터링

변환 결과 샘플을 확인하고, 너무 짧거나 원문과 동일한 경우를 필터링합니다.

In [ ]:
diary_df = pd.DataFrame(converted_rows)

print('=== 카테고리별 샘플 (각 2개) ===')
for label, cat in STAGE2_CATEGORIES.items():
    samples = diary_df[diary_df['label'] == label].head(2)
    print(f'\n[{cat}]')
    for _, row in samples.iterrows():
        print(f'  원문 : {row["original"][:60]}')
        print(f'  변환 : {row["text"][:60]}')
        print()

In [ ]:
# ── 품질 필터링 ──────────────────────────────────────
MIN_LENGTH = 10   # 최소 글자 수

before = len(diary_df)
# 1. 너무 짧은 변환 제거
diary_df = diary_df[diary_df['text'].str.len() >= MIN_LENGTH]
# 2. ok 또는 fallback만 유지 (error 제거)
diary_df = diary_df[diary_df['status'].isin(['ok', 'fallback'])]
after = len(diary_df)

print(f'필터링: {before:,} → {after:,}건 ({before - after:,}건 제거)')
print()
print('필터링 후 클래스 분포:')
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (diary_df['label'] == label).sum()
    print(f'  {cat}: {cnt:,}건')

# 저장
save_df = diary_df[['text', 'label']].copy()
save_df.to_csv(DIARY_TRAIN_PATH, index=False, encoding='utf-8')
print(f'\n✅ 저장: {DIARY_TRAIN_PATH}')

## 8. Stage 2 재학습 데이터 준비

**혼합 전략:**
- 원본(대화체) + 일기체 변환본 1:1 혼합
- Val 셋은 원본 그대로 (평가 일관성)

In [ ]:
# EEVE 언로드 (VRAM 확보)
del eeve
torch.cuda.empty_cache()
print(f'EEVE 언로드 완료. 남은 VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# 원본 + 일기체 혼합
original_df = s2_train[['text', 'label']].copy()
diary_train_df = pd.read_csv(DIARY_TRAIN_PATH)

mixed_df = pd.concat([original_df, diary_train_df], ignore_index=True)\
             .sample(frac=1, random_state=42).reset_index(drop=True)

print(f'원본       : {len(original_df):,}건')
print(f'일기체     : {len(diary_train_df):,}건')
print(f'혼합 합계  : {len(mixed_df):,}건')
print()
print('혼합 클래스 분포:')
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (mixed_df['label'] == label).sum()
    print(f'  {cat}: {cnt:,}건')

## 9. Stage 2 재학습

KURE 임베딩 (frozen) + BurnoutClassifier — FineTune_v2와 동일 구조

In [ ]:
EMBEDDING_DIM = 1024

# KURE 로드 (임베딩 전용)
print('KURE 로딩 중...')
kure = SentenceTransformer('nlpai-lab/KURE-v1', device=device)
print('✅ KURE 로드 완료')


def get_embeddings(df, cache_path):
    if os.path.exists(cache_path):
        print(f'캐시 로드: {cache_path}')
        return torch.load(cache_path, weights_only=False)
    print(f'임베딩 생성 중... ({len(df):,}건)')
    embs = kure.encode(df['text'].tolist(), batch_size=64,
                       show_progress_bar=True, convert_to_tensor=True, device=device)
    torch.save(embs.cpu(), cache_path)
    return embs.cpu()


# 임베딩 생성 (캐시 활용)
mixed_embs = get_embeddings(mixed_df, f'{DATA_PATH}/s2_st_train_embs.pt')
val_embs   = get_embeddings(s2_val,   f'{DATA_PATH}/s2_val_embs.pt')

mixed_labels = torch.tensor(mixed_df['label'].values, dtype=torch.long)
val_labels   = torch.tensor(s2_val['label'].values,   dtype=torch.long)

print(f'Train 임베딩: {mixed_embs.shape}')
print(f'Val   임베딩: {val_embs.shape}')

In [ ]:
class BurnoutClassifier(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=256, num_classes=4, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight,
                                  label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()


# 모델 초기화 + warm-start
model = BurnoutClassifier(input_dim=EMBEDDING_DIM, hidden_dim=256,
                          num_classes=4, dropout=0.3).to(device)

if os.path.exists(S2_MODEL_PATH):
    ckpt = torch.load(S2_MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'✅ warm-start: {S2_MODEL_PATH}')
else:
    print('⚠️  warm-start 모델 없음 → random init')

ST_CONFIG = {
    'epochs': 60, 'batch_size': 64, 'lr': 3e-4, 'weight_decay': 1e-4,
    'patience': 10, 'warmup_epochs': 3, 'focal_gamma': 2.0, 'label_smoothing': 0.05,
}

counts = torch.bincount(mixed_labels).float()
class_weights = (1.0 / counts).to(device)
class_weights = class_weights / class_weights.sum() * 4

criterion = FocalLoss(gamma=ST_CONFIG['focal_gamma'], weight=class_weights,
                      label_smoothing=ST_CONFIG['label_smoothing'])
optimizer = torch.optim.AdamW(model.parameters(), lr=ST_CONFIG['lr'],
                              weight_decay=ST_CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ST_CONFIG['epochs'])

print(f'클래스 가중치: {class_weights.cpu().numpy().round(3)}')

In [ ]:
train_loader = DataLoader(TensorDataset(mixed_embs, mixed_labels),
                          batch_size=ST_CONFIG['batch_size'], shuffle=True)
val_loader   = DataLoader(TensorDataset(val_embs, val_labels),
                          batch_size=256, shuffle=False)

best = {'f1': 0.0, 'acc': 0.0, 'epoch': 0}
patience_cnt = 0

print('학습 시작')
print('=' * 65)

for epoch in range(ST_CONFIG['epochs']):
    # Warmup
    if epoch < ST_CONFIG['warmup_epochs']:
        for g in optimizer.param_groups:
            g['lr'] = ST_CONFIG['lr'] * (epoch + 1) / ST_CONFIG['warmup_epochs']

    # Train
    model.train()
    train_loss = 0.0
    for emb, lbl in train_loader:
        emb, lbl = emb.to(device), lbl.to(device)
        optimizer.zero_grad()
        loss = criterion(model(emb), lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    if epoch >= ST_CONFIG['warmup_epochs']:
        scheduler.step()

    # Validate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for emb, lbl in val_loader:
            all_preds.extend(model(emb.to(device)).argmax(dim=1).cpu().tolist())
            all_labels.extend(lbl.tolist())

    val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    val_f1  = f1_score(all_labels, all_preds, average='macro')

    improved = val_f1 > best['f1']
    if improved:
        best = {'f1': val_f1, 'acc': val_acc, 'epoch': epoch + 1}
        torch.save({
            'model_state_dict': model.state_dict(),
            'embedding_dim': EMBEDDING_DIM,
            'hidden_dim': 256, 'dropout': 0.3, 'num_classes': 4,
            'categories': STAGE2_CATEGORIES,
            'config': ST_CONFIG,
            'best_metrics': best,
            'data_version': 'style_transfer_v1',
        }, SAVE_MODEL_PATH)
        patience_cnt = 0
        marker = ' ★ BEST'
    else:
        patience_cnt += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or improved:
        print(f'Epoch {epoch+1:3d} | Loss {train_loss:.4f} | '
              f'Acc {val_acc:.4f} | F1-macro {val_f1:.4f}{marker}')

    if patience_cnt >= ST_CONFIG['patience']:
        print(f'\nEarly stopping at epoch {epoch + 1}')
        break

print('=' * 65)
print(f'최고: Epoch {best["epoch"]} | F1 {best["f1"]:.4f} | Acc {best["acc"]:.4f}')

## 10. 성능 비교

In [ ]:
ckpt = torch.load(SAVE_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for emb, lbl in val_loader:
        all_preds.extend(model(emb.to(device)).argmax(dim=1).cpu().tolist())
        all_labels.extend(lbl.tolist())

print('─── Classification Report ───')
print(classification_report(
    all_labels, all_preds,
    target_names=list(STAGE2_CATEGORIES.values()), digits=4
))

final_f1 = f1_score(all_labels, all_preds, average='macro')
baseline_f1 = 0.4754  # Stage 2 v3 기준
improvement  = final_f1 - baseline_f1

print('=' * 60)
print('📊 최종 성능 비교')
print(f'  Stage 2 v3 (기준)      : F1 = {baseline_f1:.4f}')
print(f'  Stage 2 FineTune v4    : F1 = 0.4839')
print(f'  Stage 2 StyleTransfer  : F1 = {final_f1:.4f}')
print(f'  v3 대비 개선            : {improvement:+.4f}')
print()
if improvement > 0.02:
    print('  ✅ 스타일 트랜스퍼 효과 있음 → stage2_model_st.pt 사용 권장')
elif improvement > 0:
    print('  ↔ 미미한 향상 → 추가 실험 고려')
else:
    print('  ❌ 개선 없음 → 데이터 품질 점검 또는 혼합 비율 조정 필요')
print('=' * 60)

## 11. 직접 문장 테스트

특정 일기체 문장을 하드코딩해서 예측 결과를 바로 확인합니다.
이전 FineTune_v2에서 오분류됐던 케이스 포함.

In [ ]:
# ── 직접 문장 테스트 ─────────────────────────────────────
# 이전에 오분류됐던 일기체 케이스 + 추가 테스트 문장
TEST_SENTENCES = [
    ("잠을 못 잤더니 지쳤다",         "정서적_고갈"),   # 이전 오분류: 자기비하
    ("오늘도 야근. 몸이 한계다",        "정서적_고갈"),
    ("팀장이 또 뭐라 했다. 짜증",       "좌절_압박"),
    ("왜 나만 이렇게 힘들까. 내 탓인가", "자기비하"),
    ("동료가 내 공을 가로챘다. 억울",   "부정적_대인관계"),
    ("아무것도 하기 싫다. 그냥 누워있고 싶어", "정서적_고갈"),
]

LABEL_MAP = {v: k for k, v in STAGE2_CATEGORIES.items()}  # 한국어→숫자

model.eval()
print(f"[StyleTransfer 모델 — {SAVE_MODEL_PATH}]
")
print(f"  {"문장":<30} {"예측"}  {"정답"}  {"신뢰도"}  {"OK?"}")
print("-" * 70)

with torch.no_grad():
    for text, gt in TEST_SENTENCES:
        emb = torch.tensor(
            kure.encode([text], normalize_embeddings=True),
            dtype=torch.float32
        ).to(device)
        logits = model(emb)
        probs  = torch.softmax(logits, dim=1)[0]
        pred_idx = probs.argmax().item()
        pred_cat = STAGE2_CATEGORIES[pred_idx]
        conf     = probs[pred_idx].item()
        ok       = "✅" if pred_cat == gt else "❌"
        print(f"  {text:<30} {pred_cat:<12} {gt:<12} {conf:.1%}  {ok}")
